In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pathlib
import warnings
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
from lmfit import Model
from scipy.stats import pearsonr
from sklearn.cross_decomposition import PLSRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# ===================== CGD Journal Style: Times New Roman, English Only =====================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.formatter.useoffset'] = False

# Font size & bold for journal figures
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# XRD physical constants
XRD_LAMBDA = 1.5406    # Cu Kα wavelength (Å)
SCHERRER_K = 0.9
FWHM_INST = 0.05       # Instrumental broadening (°)
MIN_GOOD_SAMPLES = 3

# Folder definitions
PLOT_FOLDER = "DATA"
os.makedirs(PLOT_FOLDER, exist_ok=True)

# Range definition
SUB_MASK_RANGE = (41.5, 42.0)
FIT_RANGE = (37.0, 41.0)

# ===================== Phase peak positions (2θ, degree) =====================
_ALPHA_PEAKS = np.array([39.9, 40.2])
_BETA_PEAKS  = np.array([38.388])
_KAPPA_PEAKS = np.array([38.769])
_ALL_PEAKS_FULL = sorted(np.concatenate([_ALPHA_PEAKS, _BETA_PEAKS, _KAPPA_PEAKS]))
ALL_PEAKS = np.array([p for p in _ALL_PEAKS_FULL if FIT_RANGE[0] <= p <= FIT_RANGE[1]])

def label_phase(peak):
    """Map peak position to phase index: 0=α,1=β,2=κ"""
    if peak in _ALPHA_PEAKS:
        return 0
    elif peak in _BETA_PEAKS:
        return 1
    else:
        return 2

PHASE_OF_PEAK = [label_phase(p) for p in ALL_PEAKS]
ALPHA_IDX = [i for i, ph in enumerate(PHASE_OF_PEAK) if ph == 0]
BETA_IDX  = [i for i, ph in enumerate(PHASE_OF_PEAK) if ph == 1]
KAPPA_IDX = [i for i, ph in enumerate(PHASE_OF_PEAK) if ph == 2]

def get_peak_bounds(peak, phase):
    """Define fitting boundary for each peak according to phase"""
    if phase == 0:
        return (peak - 0.20, peak + 0.20)
    elif phase == 1:
        return (peak - 0.25, peak + 0.25)
    else:
        return (peak - 0.50, peak + 0.50)

PEAK_BOUNDS = [get_peak_bounds(ALL_PEAKS[i], PHASE_OF_PEAK[i]) for i in range(len(ALL_PEAKS))]


In [2]:
# ===================== Basic physical & fitting functions =====================
def voigt_func(x, amp, center, sigma, gamma):
    """Voigt profile: convolution of Gaussian + Lorentzian"""
    g = np.exp(-(x - center)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))
    l = gamma / (np.pi * ((x - center)**2 + gamma**2))
    return amp * (0.5 * g + 0.5 * l)
def calc_voigt_fwhm(sigma, gamma):
    """Calculate FWHM from Voigt parameters"""
    fwhm_g = 2 * sigma * np.sqrt(2 * np.log(2))
    fwhm_l = 2 * gamma
    return 0.5346 * fwhm_l + np.sqrt(0.2166 * fwhm_l**2 + fwhm_g**2)
def subtract_background(x, y, poly_order=3, iterations=5):
    """Iterative polynomial background subtraction"""
    y_ = y.copy()
    for _ in range(iterations):
        p = np.polyfit(x, y_, poly_order)
        bg = np.polyval(p, x)
        y_ = np.minimum(y_, bg)
    p = np.polyfit(x, y_, poly_order)
    bg = np.polyval(p, x)
    return y - bg, bg
def load_xrd_data(folder, pattern):
    """Load all XRD txt files, align theta axis"""
    intensity_list, params, thetas = [], [], []
    for f in os.listdir(folder):
        if f.endswith(".txt"):
            match = re.match(pattern, f)
            if match:
                ar, p, si, t = int(match[1]), int(match[2]), int(match[3]), int(match[4])
                params.append([ar, p, si, t])
                data = np.loadtxt(os.path.join(folder, f), delimiter=',', comments='#')
                thetas.append(data[:, 0])
                intensity_list.append(data[:, 1])
    x_ref = thetas[0]
    aligned = []
    for th, i in zip(thetas, intensity_list):
        if not np.allclose(th, x_ref):
            aligned.append(np.interp(x_ref, th, i))
        else:
            aligned.append(i)
    return x_ref, aligned, np.array(params)

In [3]:
# ===================== Fitting functions =====================
def fit_free_no_bg(x, y):
    try:
        model = None
        for i, cen in enumerate(ALL_PEAKS):
            m = Model(voigt_func, prefix=f"p{i}_")
            model = m if model is None else model + m
        model += Model(lambda x, b0, b1: b0 + b1*x, prefix="bg_")
        params = model.make_params()
        for i, cen in enumerate(ALL_PEAKS):
            low, high = PEAK_BOUNDS[i]
            params.add(f"p{i}_amp", value=np.max(y)*0.2, min=0)
            params.add(f"p{i}_center", value=cen, min=low, max=high)
            params.add(f"p{i}_sigma", value=0.1, min=0.03, max=0.5)
            params.add(f"p{i}_gamma", value=0.1, min=0.03, max=0.5)
        params.add("bg_b0", value=np.min(y), min=0)
        params.add("bg_b1", value=0, min=-0.1, max=0.1)
        res = model.fit(y, params, x=x, method='leastsq', nan_policy='omit')
        fit_y = res.best_fit
        rwp = np.sqrt(np.sum((y-fit_y)**2)/np.sum(y**2))*100
        peak_fwhms = [calc_voigt_fwhm(res.params[f"p{i}_sigma"].value, res.params[f"p{i}_gamma"].value) for i in range(len(ALL_PEAKS))]
        def phase_params(idx_list):
            if not idx_list:
                return 0.0, 0.0
            amps = [res.params[f"p{ii}_amp"].value for ii in idx_list]
            total_amp = sum(amps)
            if total_amp <= 0:
                return 0.0, 0.0
            return total_amp, total_amp
        aa, _ = phase_params(ALPHA_IDX)
        ab, _ = phase_params(BETA_IDX)
        ak, _ = phase_params(KAPPA_IDX)
        tot = aa + ab + ak + 1e-8
        return {
            "fit_y": fit_y, "RWP": rwp,
            "ratio_alpha": aa/tot, "ratio_beta": ab/tot, "ratio_kappa": ak/tot,
            "amp_alpha": aa, "amp_beta": ab, "amp_kappa": ak, "params": res.params,
            "sigma": [res.params[f"p{i}_sigma"].value for i in range(len(ALL_PEAKS))],
            "gamma": [res.params[f"p{i}_gamma"].value for i in range(len(ALL_PEAKS))]
        }
    except Exception as e:
        warnings.warn(f"Fit no-bg failed: {str(e)}")
        return None
def fit_free_with_bg(x, y):
    try:
        y_sub, bg = subtract_background(x, y)
        if np.max(y_sub) < 1e-3:
            return None
        model = None
        for i, cen in enumerate(ALL_PEAKS):
            m = Model(voigt_func, prefix=f"p{i}_")
            model = m if model is None else model + m
        params = model.make_params()
        for i, cen in enumerate(ALL_PEAKS):
            low, high = PEAK_BOUNDS[i]
            params.add(f"p{i}_amp", value=np.max(y_sub)*0.2, min=0)
            params.add(f"p{i}_center", value=cen, min=low, max=high)
            params.add(f"p{i}_sigma", value=0.1, min=0.03, max=0.5)
            params.add(f"p{i}_gamma", value=0.1, min=0.03, max=0.5)
        res = model.fit(y_sub, params, x=x, method='leastsq', nan_policy='omit')
        fit_y = res.best_fit + bg
        rwp = np.sqrt(np.sum((y-fit_y)**2)/np.sum(y**2))*100
        peak_fwhms = [calc_voigt_fwhm(res.params[f"p{i}_sigma"].value, res.params[f"p{i}_gamma"].value) for i in range(len(ALL_PEAKS))]
        def phase_params(idx_list):
            if not idx_list:
                return 0.0, 0.0
            amps = [res.params[f"p{ii}_amp"].value for ii in idx_list]
            total_amp = sum(amps)
            if total_amp <= 0:
                return 0.0, 0.0
            return total_amp, total_amp
        aa, _ = phase_params(ALPHA_IDX)
        ab, _ = phase_params(BETA_IDX)
        ak, _ = phase_params(KAPPA_IDX)
        tot = aa + ab + ak + 1e-8
        return {
            "fit_y": fit_y, "RWP": rwp,
            "ratio_alpha": aa/tot, "ratio_beta": ab/tot, "ratio_kappa": ak/tot,
            "amp_alpha": aa, "amp_beta": ab, "amp_kappa": ak, "params": res.params,
            "sigma": [res.params[f"p{i}_sigma"].value for i in range(len(ALL_PEAKS))],
            "gamma": [res.params[f"p{i}_gamma"].value for i in range(len(ALL_PEAKS))]
        }
    except Exception as e:
        warnings.warn(f"Fit with-bg failed: {str(e)}")
        return None
def fit_with_good_init(x, y, good_sigma, good_gamma):
    try:
        model = None
        for i, cen in enumerate(ALL_PEAKS):
            m = Model(voigt_func, prefix=f"p{i}_")
            model = m if model is None else model + m
        model += Model(lambda x, b0, b1: b0 + b1*x, prefix="bg_")
        params = model.make_params()
        for i, cen in enumerate(ALL_PEAKS):
            low, high = PEAK_BOUNDS[i]
            params.add(f"p{i}_amp", value=np.max(y)*0.2, min=0)
            params.add(f"p{i}_center", value=cen, min=low, max=high)
            params.add(f"p{i}_sigma", value=good_sigma[i], min=0.03, max=0.8)
            params.add(f"p{i}_gamma", value=good_gamma[i], min=0.03, max=0.8)
        params.add("bg_b0", value=np.min(y), min=0)
        params.add("bg_b1", value=0, min=-0.1, max=0.1)
        res = model.fit(y, params, x=x, method='leastsq', nan_policy='omit')
        fit_y = res.best_fit
        rwp = np.sqrt(np.sum((y-fit_y)**2)/np.sum(y**2))*100
        aa = sum(res.params[f"p{i}_amp"].value for i in ALPHA_IDX)
        ab = sum(res.params[f"p{i}_amp"].value for i in BETA_IDX)
        ak = sum(res.params[f"p{i}_amp"].value for i in KAPPA_IDX)
        tot = aa + ab + ak + 1e-8
        return {
            "fit_y": fit_y, "RWP": rwp,
            "ratio_alpha": aa/tot, "ratio_beta": ab/tot, "ratio_kappa": ak/tot,
            "amp_alpha": aa, "amp_beta": ab, "amp_kappa": ak
        }
    except Exception as e:
        warnings.warn(f"Guided fit failed: {str(e)}")
        return None

In [17]:
# ===================== Plot & Export functions =====================
def save_fig(filename):
    """
    Export figure for journal submission:
        PNG: 1200 dpi (for manuscript review)
        PDF: vector graphic (for final publication)
    """
    png_path = os.path.join(PLOT_FOLDER, f"{filename}.png")
    pdf_path = os.path.join(PLOT_FOLDER, f"{filename}.pdf")
    plt.savefig(png_path, dpi=1200, bbox_inches='tight')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.close()

def format_si_label(si_val):
    """将真实Si浓度格式化为科学计数法显示，0直接返回0"""
    if si_val == 0:
        return "0"
    exp = int(np.floor(np.log10(abs(si_val))))
    coeff = si_val / (10 ** exp)
    # 自动去除末尾无效0，保证显示简洁
    if abs(coeff - round(coeff)) < 1e-6:
        coeff_str = f"{int(round(coeff))}"
    else:
        coeff_str = f"{coeff:.3f}".rstrip('0').rstrip('.')
    return f"{coeff_str}$\\times10^{{{exp}}}$"

def plot_fit_only(x, y, final_res, idx, param):
    si_real = param[2]  # 真实浓度，单位μmol/min
    si_label = format_si_label(si_real)
    plt.figure(figsize=(7, 4.5))
    plt.plot(x, y, 'k-', label="Raw XRD", lw=1.2)
    plt.plot(x, final_res["fit_y"], 'r-', lw=2.0, label="Fitting")
    plt.title(f"Sample {idx+1} | Ar={param[0]}, P={param[1]} mbar, Si={si_label} μmol/min, T={param[3]} s")
    plt.xlabel("2θ (°)")
    plt.ylabel("Intensity (a.u.)")
    plt.legend(loc='upper left')
    plt.grid(alpha=0.3, linestyle='--')
    save_fig(f"Sample_{idx+1}")

def plot_sample9_10_compare(x_full, y_list_full):
    idx9 = 8
    idx10 = 9
    mask = (x_full >= FIT_RANGE[0]) & (x_full <= FIT_RANGE[1])
    x = x_full[mask]
    y9 = np.asarray(y_list_full[idx9])[mask]
    y10 = np.asarray(y_list_full[idx10])[mask]
    plt.figure(figsize=(7, 4.5))
    plt.plot(x, y9, 'k-', label="Sample 9", lw=1.2)
    plt.plot(x, y10, 'r-', label="Sample 10", lw=1.2)
    plt.xlabel("2θ (°)")
    plt.ylabel("Intensity (a.u.)")
    plt.title("XRD comparison of Sample 9 and Sample 10 (37°–41°)")
    plt.legend(loc='upper left')
    plt.grid(alpha=0.3, linestyle='--')
    save_fig("Sample9_10_compare")

In [18]:
# ===================== Main workflow =====================
def time_to_thickness_group(time_val):
    """Classify sample by growth time (thin/thick film)"""
    return "Thin film" if time_val <= 4000 else "Thick film"

if __name__ == "__main__":
    DATA_FOLDER = "xrd_data"
    FILE_REGEX = r"TE_Ar(\d+)_P(\d+)mbar_Si(\d+)_T(\d+)s\.txt"
    print("🔍 Loading XRD raw data...")
    x_full, y_list_full, params = load_xrd_data(DATA_FOLDER, FILE_REGEX)

    # ========== 关键修复：转为浮点型数组，避免整数截断 ==========
    params = params.astype(float)

    # 文件名Si编号 → 真实浓度 (μmol/min) 换算映射
    si_conversion_map = {
        0: 0.0,
        5: 0.0005,
        10: 0.001,
        30: 0.0029,
        40: 0.00371
    }
    # 批量转换所有样品的Si浓度为真实物理值
    for p in params:
        si_code = int(p[2])
        p[2] = si_conversion_map.get(si_code, si_code)

    print(f"✅ Total {len(y_list_full)} samples loaded")
    mask = (x_full >= FIT_RANGE[0]) & (x_full <= FIT_RANGE[1])
    x, y_list = x_full[mask], [y[mask] for y in y_list_full]
    # Fixed bug: original sub_mask logic error
    sub_mask = (x_full >= SUB_MASK_RANGE[0]) & (x_full <= SUB_MASK_RANGE[1])
    sample_info, all_best_init = [], []
    print("\n========== First round fitting ==========")
    for idx, (y, p) in enumerate(zip(y_list, params)):
        res_nobg = fit_free_no_bg(x, y)
        res_wbg = fit_free_with_bg(x, y)
        rwp2 = res_wbg["RWP"] if res_wbg else 999
        if res_nobg is None and res_wbg is None:
            warnings.warn(f"Sample {idx+1}: both fitting failed")
            sample_info.append({"nobg_res": None, "wbg_res": None, "best_init": None, "p": p})
            all_best_init.append(None)
            continue
        elif res_nobg is None:
            best_init = res_wbg
        elif res_wbg is None:
            best_init = res_nobg
        else:
            best_init = res_nobg if res_nobg["RWP"] <= rwp2 else res_wbg
        sample_info.append({"nobg_res": res_nobg, "wbg_res": res_wbg, "best_init": best_init, "p": p})
        all_best_init.append(best_init)
        print(f"Sample {idx+1} best RWP: {best_init['RWP']:.2f}%")
    valid_samples = [s for s in all_best_init if s is not None]
    if len(valid_samples) >= MIN_GOOD_SAMPLES:
        good_samples = sorted(valid_samples, key=lambda x: x["RWP"])[:MIN_GOOD_SAMPLES]
        good_sigma = np.mean([s["sigma"] for s in good_samples], axis=0)
        good_gamma = np.mean([s["gamma"] for s in good_samples], axis=0)
        print(f"\n✅ Good-sample peak shape parameters loaded")
    else:
        raise RuntimeError("Not enough valid fitted samples for initial guess calibration")
    final_result_list = []
    print("\n========== Final guided fitting ==========")
    for idx, info in enumerate(sample_info):
        y_raw, p = y_list[idx], info["p"]
        init_best = info["best_init"]
        thick_group = time_to_thickness_group(p[3])
        if init_best is None:
            final_result_list.append({
                "Sample": idx + 1,
                "Ar(sccm)": p[0],
                "P(mbar)": p[1],
                "Si(μmol/min)": p[2],
                "Epitaxial growth time(s)": p[3],
                "Thickness groups": thick_group,
                "Final RWP": np.nan,
                "α-phase fraction": np.nan,
                "β-phase fraction": np.nan,
                "κ-phase fraction": np.nan,
                "Status": "Fit Failed"
            })
            continue
        guided_res = fit_with_good_init(x, y_raw, good_sigma, good_gamma)
        final_res = guided_res if (guided_res is not None and guided_res["RWP"] < init_best["RWP"]) else init_best
        ratio_alpha = round(final_res["ratio_alpha"], 3)
        ratio_beta = round(final_res["ratio_beta"], 3)
        ratio_kappa = round(final_res["ratio_kappa"], 3)
        plot_fit_only(x, y_raw, final_res, idx, p)
        final_result_list.append({
            "Sample": idx + 1,
            "Ar(sccm)": p[0],
            "P(mbar)": p[1],
            "Si(μmol/min)": p[2],
            "Epitaxial growth time(s)": p[3],
            "Thickness groups": thick_group,
            "Final RWP": round(final_res["RWP"], 2),
            "α-phase fraction": ratio_alpha,
            "β-phase fraction": ratio_beta,
            "κ-phase fraction": ratio_kappa,
            "Status": "Success"
        })
    df = pd.DataFrame(final_result_list)
    csv_path = os.path.join(PLOT_FOLDER, "XRD_final_results.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV summary table saved: {csv_path}")
    plot_sample9_10_compare(x_full, y_list_full)
    print("✅ Sample 9 & Sample 10 comparison plot saved")
    print("\n🎉 All calculation and plotting finished.")

🔍 Loading XRD raw data...
✅ Total 16 samples loaded

========== First round fitting ==========
Sample 1 best RWP: 7.81%
Sample 2 best RWP: 5.81%
Sample 3 best RWP: 6.15%
Sample 4 best RWP: 5.79%
Sample 5 best RWP: 6.91%
Sample 6 best RWP: 8.39%
Sample 7 best RWP: 4.69%
Sample 8 best RWP: 6.82%
Sample 9 best RWP: 8.86%
Sample 10 best RWP: 7.99%
Sample 11 best RWP: 11.87%
Sample 12 best RWP: 4.36%
Sample 13 best RWP: 19.75%
Sample 14 best RWP: 8.24%
Sample 15 best RWP: 14.97%
Sample 16 best RWP: 12.47%

✅ Good-sample peak shape parameters loaded

========== Final guided fitting ==========

✅ CSV summary table saved: DATA\XRD_final_results.csv
✅ Sample 9 & Sample 10 comparison plot saved

🎉 All calculation and plotting finished.


In [20]:
# =============================================================================
# Plot Fig1 main figure and TOC graphic for journal submission
# Data source: XRD_final_results.csv stored in ./DATA
# Output: PNG (1200 dpi) + vector PDF saved into ./DATA folder
# Font style, size and bold settings consistent with phase fraction plot
# =============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -------------------------- Journal Figure Style Settings --------------------------
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 1200
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.linewidth'] = 1.0

# Font size & bold configuration, fully consistent with phase fraction figure
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16

# -------------------------- Relative File Path Configuration --------------------------
DATA_FOLDER = "DATA"
os.makedirs(DATA_FOLDER, exist_ok=True)
CSV_PATH = os.path.join(DATA_FOLDER, "XRD_final_results.csv")

# Load dataset
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
df['Thickness groups'] = df['Thickness groups'].astype(str).str.strip()

# Axis label definitions and x coordinates
x_full_labels = ["20 mbar-40 sccm", "20 mbar-60 sccm", "20 mbar-80 sccm", "40 mbar-80 sccm"]
x_toc_labels  = ["20-40", "20-60", "20-80", "40-80"]
x_positions = np.array([0, 1.6, 3.2, 4.8])
y_lim_plot = [-0.08, 0.42]

# Phase color mapping
phase_color = {
    'α': '#c0392b',
    'β': '#27ae60',
    'κ': '#2980b9'
}

# Threshold for phase identification
PURE_THRESHOLD = 0.90
EXIST_THRESHOLD = 0.10

# -------------------------- Phase judgement helper functions --------------------------
def filter_phases(fa, fb, fk):
    """Filter phases with fraction above existence threshold"""
    phases = {'α': fa, 'β': fb, 'κ': fk}
    valid = {k: v for k, v in phases.items() if v >= EXIST_THRESHOLD}
    return valid

def is_pure_phase(fa, fb, fk):
    """Judge whether the sample is single-phase dominated"""
    return max(fa, fb, fk) >= PURE_THRESHOLD

def get_dominant_phase(fa, fb, fk):
    """Return the phase with maximum fraction"""
    frac = {'α': fa, 'β': fb, 'κ': fk}
    return max(frac, key=frac.get)

def get_phase_label(fa, fb, fk):
    """Generate combined label for mixed phases"""
    valid = filter_phases(fa, fb, fk)
    order = ['κ', 'α', 'β']
    sorted_phases = [p for p in order if p in valid]
    return '+'.join(sorted_phases)

def mix_phase_color(fa, fb, fk):
    """Calculate blended RGB color for mixed-phase samples"""
    valid = filter_phases(fa, fb, fk)
    total = sum(valid.values())
    r, g, b = 0.12, 0.12, 0.12
    if 'α' in valid:
        r += (valid['α'] / total) * 0.82
    if 'β' in valid:
        g += (valid['β'] / total) * 0.82
    if 'κ' in valid:
        b += (valid['κ'] / total) * 0.82
    r = min(max(r, 0), 1)
    g = min(max(g, 0), 1)
    b = min(max(b, 0), 1)
    return (r, g, b)

# -------------------------- Main Figure (Fig1) --------------------------
def plot_main_optimized():
    fig, ax = plt.subplots(figsize=(14, 6.8))
    all_points = []
    pair_dict = {}

    for x_idx, cond_label in enumerate(x_full_labels):
        p_str, ar_str = cond_label.split("-")
        p_val = float(p_str.replace(" mbar", ""))
        ar_val = float(ar_str.replace(" sccm", ""))
        # 兼容列名可能的拼写差异，确保所有样品都被匹配
        sub_df = df[np.isclose(df['P(mbar)'], p_val) & np.isclose(df['Ar(sccm)'], ar_val)]

        for _, row in sub_df.iterrows():
            si_raw = row['Si(μmol/min)']
            y_plot = si_raw * 100
            fa = row['α-phase fraction']
            fb = row['β-phase fraction']
            fk = row['κ-phase fraction']

            dominant = get_dominant_phase(fa, fb, fk)
            pure = is_pure_phase(fa, fb, fk)
            thick = row['Thickness groups']

            halo_color = None
            if not pure:
                halo_color = mix_phase_color(fa, fb, fk)

            # 恢复薄厚x轴偏移，避免点重叠
            if "Thin film" in thick:
                x_plot = x_positions[x_idx] - 0.20
            else:
                x_plot = x_positions[x_idx] + 0.20

            point_info = {
                'x': x_plot,
                'y': y_plot,
                'dominant': dominant,
                'thick': thick,
                'pure': pure,
                'halo_color': halo_color
            }
            all_points.append(point_info)

            # 配对薄厚样品，用于画箭头
            pair_key = (x_idx, round(si_raw, 6))
            if pair_key not in pair_dict:
                pair_dict[pair_key] = {'thin': None, 'thick': None}
            if "Thin film" in thick:
                pair_dict[pair_key]['thin'] = (x_plot, y_plot)
            else:
                pair_dict[pair_key]['thick'] = (x_plot, y_plot)

    # 绘制混合相光晕背景
    for pt in all_points:
        if not pt['pure'] and pt['halo_color'] is not None:
            ax.scatter(
                pt['x'], pt['y'],
                s=500,
                color=pt['halo_color'],
                alpha=0.45,
                marker='o',
                zorder=2
            )

    # 绘制薄→厚箭头
    for pair_key, pos in pair_dict.items():
        p_thin = pos['thin']
        p_thick = pos['thick']
        if p_thin is not None and p_thick is not None:
            ax.annotate(
                '', xy=p_thick, xytext=p_thin,
                arrowprops=dict(arrowstyle='->', color='#444444', lw=1.2),
                zorder=3
            )

    # 绘制数据点marker
    for pt in all_points:
        c = phase_color[pt['dominant']]
        if "Thin film" in pt['thick']:
            ax.scatter(
                pt['x'], pt['y'],
                marker='o', facecolor='white', edgecolor=c,
                s=110, linewidth=1.8, zorder=5
            )
        else:
            ax.scatter(
                pt['x'], pt['y'],
                marker='s', facecolor=c, edgecolor='black',
                s=110, linewidth=1.0, zorder=5
            )

    ax.axhline(y=0, color='black', linewidth=0.9, linestyle='--', zorder=1)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_full_labels, rotation=0, ha='center')
    ax.set_xlabel("Growth condition (Vapor-phase supersaturation)", labelpad=12)
    ax.set_ylabel(r"Si doping flux ($\times10^{-2}\ \mu\mathrm{mol/min}$)", labelpad=12)
    ax.set_ylim(y_lim_plot)
    ax.set_xlim(-0.7, x_positions[-1] + 1.0)
    ax.grid(axis='y', alpha=0.3, linestyle='--', zorder=0)

    # 加粗刻度标签
    ax.tick_params(axis='both', which='major', labelsize=17)
    for label in ax.get_xticklabels():
        label.set_weight('bold')
    for label in ax.get_yticklabels():
        label.set_weight('bold')

    # 自定义图例：仅去掉厚度数值后缀，保留薄厚区分
    legend_handles = [
        Line2D([], [], marker='o', color='w', markerfacecolor='white', markeredgecolor='black',
               markersize=10, linewidth=0, label='Thin film'),
        Line2D([], [], marker='s', color='w', markerfacecolor='black', markeredgecolor='black',
               markersize=10, linewidth=0, label='Thick film'),
        Line2D([], [], marker='', color='#444444', lw=1.2, label='Thin → Thick'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#c0392b', markeredgecolor='black',
               markersize=10, linewidth=0, label=r'$\alpha$-dominant'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#27ae60', markeredgecolor='black',
               markersize=10, linewidth=0, label=r'$\beta$-dominant'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#2980b9', markeredgecolor='black',
               markersize=10, linewidth=0, label=r'$\kappa$-dominant'),
        Line2D([], [], marker='o', color='w', markerfacecolor='gray', markeredgecolor='gray',
               markersize=14, alpha=0.45, linewidth=0, label='Mixed phase region'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', frameon=False, bbox_to_anchor=(1, 1),
              prop={'weight':'bold'})

    plt.tight_layout()
    save_png = os.path.join(DATA_FOLDER, "Fig1_main_optimized.png")
    save_pdf = os.path.join(DATA_FOLDER, "Fig1_main_optimized.pdf")
    plt.savefig(save_png, bbox_inches='tight')
    plt.savefig(save_pdf, bbox_inches='tight')
    plt.close()
    print(f"✅ Main figure (Fig1) saved\n{save_png}\n{save_pdf}")

# -------------------------- TOC Graph --------------------------
def plot_toc_simplified():
    fig, ax = plt.subplots(figsize=(8.5, 5.0))
    all_points = []
    pair_dict = {}

    for x_idx, cond_label in enumerate(x_full_labels):
        p_str, ar_str = cond_label.split("-")
        p_val = float(p_str.replace(" mbar", ""))
        ar_val = float(ar_str.replace(" sccm", ""))
        sub_df = df[np.isclose(df['P(mbar)'], p_val) & np.isclose(df['Ar(sccm)'], ar_val)]

        for _, row in sub_df.iterrows():
            si_raw = row['Si(μmol/min)']
            y_plot = si_raw * 100
            fa = row['α-phase fraction']
            fb = row['β-phase fraction']
            fk = row['κ-phase fraction']

            dominant = get_dominant_phase(fa, fb, fk)
            pure = is_pure_phase(fa, fb, fk)
            thick = row['Thickness groups']

            halo_color = None
            if not pure:
                halo_color = mix_phase_color(fa, fb, fk)

            if "Thin film" in thick:
                x_plot = x_positions[x_idx] - 0.20
            else:
                x_plot = x_positions[x_idx] + 0.20

            all_points.append({
                'x': x_plot,
                'y': y_plot,
                'dominant': dominant,
                'thick': thick,
                'pure': pure,
                'halo_color': halo_color
            })

            pair_key = (x_idx, round(si_raw, 6))
            if pair_key not in pair_dict:
                pair_dict[pair_key] = {'thin': None, 'thick': None}
            if "Thin film" in thick:
                pair_dict[pair_key]['thin'] = (x_plot, y_plot)
            else:
                pair_dict[pair_key]['thick'] = (x_plot, y_plot)

    # 混合相光晕
    for pt in all_points:
        if not pt['pure'] and pt['halo_color'] is not None:
            ax.scatter(pt['x'], pt['y'], s=420, color=pt['halo_color'], alpha=0.45, marker='o', zorder=2)

    # 薄→厚箭头
    for pair_key, pos in pair_dict.items():
        p_thin = pos['thin']
        p_thick = pos['thick']
        if p_thin is not None and p_thick is not None:
            ax.annotate('', xy=p_thick, xytext=p_thin,
                        arrowprops=dict(arrowstyle='->', color='#444444', lw=1.1), zorder=3)

    # 绘制数据点
    for pt in all_points:
        c = phase_color[pt['dominant']]
        if "Thin film" in pt['thick']:
            ax.scatter(pt['x'], pt['y'], marker='o', facecolor='white', edgecolor=c,
                       s=95, linewidth=1.7, zorder=5)
        else:
            ax.scatter(pt['x'], pt['y'], marker='s', facecolor=c, edgecolor='black',
                       s=95, linewidth=0.9, zorder=5)

    ax.axhline(y=0, color='black', linewidth=0.9, linestyle='--', zorder=1)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_toc_labels, rotation=0, ha='center')
    ax.set_xlabel("Growth condition", labelpad=12)
    ax.set_ylabel(r"Si-flux ($\times10^{-2}\ \mu\mathrm{mol/min}$)", labelpad=12)
    ax.set_ylim(y_lim_plot)
    ax.set_xlim(-0.7, x_positions[-1] + 0.7)
    ax.grid(axis='y', alpha=0.3, linestyle='--', zorder=0)

    # 加粗刻度标签
    ax.tick_params(axis='both', which='major', labelsize=17)
    for label in ax.get_xticklabels():
        label.set_weight('bold')
    for label in ax.get_yticklabels():
        label.set_weight('bold')

    # TOC图例：去掉厚度数值
    legend_handles = [
        Line2D([], [], marker='o', color='w', markerfacecolor='white', markeredgecolor='black',
               markersize=9, linewidth=0, label='Thin film'),
        Line2D([], [], marker='s', color='w', markerfacecolor='black', markeredgecolor='black',
               markersize=9, linewidth=0, label='Thick film'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#c0392b', markeredgecolor='black',
               markersize=9, linewidth=0, label=r'$\alpha$'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#27ae60', markeredgecolor='black',
               markersize=9, linewidth=0, label=r'$\beta$'),
        Line2D([], [], marker='o', color='w', markerfacecolor='#2980b9', markeredgecolor='black',
               markersize=9, linewidth=0, label=r'$\kappa$'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', frameon=False, prop={'weight':'bold'})

    plt.tight_layout()
    save_png = os.path.join(DATA_FOLDER, "Fig1_TOC_simplified.png")
    save_pdf = os.path.join(DATA_FOLDER, "Fig1_TOC_simplified.pdf")
    plt.savefig(save_png, bbox_inches='tight')
    plt.savefig(save_pdf, bbox_inches='tight')
    plt.close()
    print(f"✅ TOC graphic saved\n{save_png}\n{save_pdf}")

if __name__ == "__main__":
    plot_main_optimized()
    plot_toc_simplified()
    print("\n🎉 All figures generated successfully")


✅ Main figure (Fig1) saved
DATA\Fig1_main_optimized.png
DATA\Fig1_main_optimized.pdf
✅ TOC graphic saved
DATA\Fig1_TOC_simplified.png
DATA\Fig1_TOC_simplified.pdf

🎉 All figures generated successfully


In [21]:
# ============================================================
# Generate publication-quality figure from XRD_final_results.csv
# Fig2: Phase fraction of three phases under different growth conditions (thin/thick film comparison)
# All input/output files inside DATA folder
# Font settings match Fig1: labelsize=18, tick=17, legend=16; all text bold
# ============================================================
# ---------- Journal global style (consistent with Fig1) ----------
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.formatter.useoffset'] = False

# ========= Font size & bold setting, consistent with Fig1 =========
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.linewidth'] = 0.8

# ---------- Path configuration ----------
DATA_FOLDER = "./DATA"
os.makedirs(DATA_FOLDER, exist_ok=True)
CSV_PATH = os.path.join(DATA_FOLDER, "XRD_final_results.csv")
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV file missing: {CSV_PATH}\nRun upstream fitting code first to generate this csv!")

# Phase color scheme: α red / β green / κ blue
COLOR_PHASE = {'α': '#e74c3c', 'β': '#2ecc71', 'κ': '#3498db'}
# Marker style: thin = circle, thick = square
MARKER_STYLE = {'Thin film': 'o', 'Thick film': 's'}

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

def save_fig(filename):
    """Save PNG (1200 dpi) and vector PDF into DATA folder"""
    out_png = os.path.join(DATA_FOLDER, f"{filename}.png")
    out_pdf = os.path.join(DATA_FOLDER, f"{filename}.pdf")
    plt.savefig(out_png, dpi=1200, bbox_inches='tight')
    plt.savefig(out_pdf, bbox_inches='tight')
    plt.close()

# ============================================================
# Fig2: Phase fraction comparison (matches reference style)
# ============================================================
def plot_phase_transition(df):
    # Filter undoped samples (Si=0)
    df = df[np.isclose(df['Si(μmol/min)'], 0)].copy()
    
    # Group by growth conditions, sort to ensure correct order
    group_cols = ['P(mbar)', 'Ar(sccm)']
    conditions = (df.groupby(group_cols).size()
                  .reset_index().sort_values(group_cols)
                  .reset_index(drop=True))
    n_cond = len(conditions)

    fig, ax = plt.subplots(figsize=(8.8, 6.2))
    frac_cols = {'α': 'α-phase fraction',
                 'β': 'β-phase fraction',
                 'κ': 'κ-phase fraction'}
    x_pos = np.arange(n_cond)
    # Horizontal offset for three phases to avoid overlap
    phase_offset = {'α': -0.22, 'β': 0.0, 'κ': 0.22}

    # Plot each phase: thin (circle) + thick (square) + dashed connecting arrow
    for ph in ['α', 'β', 'κ']:
        col = frac_cols[ph]
        color = COLOR_PHASE[ph]
        
        for i, cond in conditions.iterrows():
            mask_cond = ((df['P(mbar)'] == cond['P(mbar)']) &
                         (df['Ar(sccm)'] == cond['Ar(sccm)']))
            sub = df[mask_cond]
            
            y_thin, y_thick = np.nan, np.nan
            for _, row in sub.iterrows():
                val = row[col] * 100
                if row['Thickness groups'] == 'Thin film':
                    y_thin = val
                elif row['Thickness groups'] == 'Thick film':
                    y_thick = val
            
            base_x = x_pos[i] + phase_offset[ph]
            
            # Plot thin film (solid circle)
            if not np.isnan(y_thin):
                ax.scatter(base_x, y_thin,
                           c=color, marker=MARKER_STYLE['Thin film'],
                           s=70, edgecolors='black', linewidths=0.8, zorder=5,
                           label=f"{ph}-phase (thin)" if i == 0 else "")
            
            # Plot thick film (solid square)
            if not np.isnan(y_thick):
                ax.scatter(base_x, y_thick,
                           c=color, marker=MARKER_STYLE['Thick film'],
                           s=70, edgecolors='black', linewidths=0.8, zorder=5,
                           label=f"{ph}-phase (thick)" if i == 0 else "")
            
            # Draw dashed arrow from thin to thick
            if not (np.isnan(y_thin) or np.isnan(y_thick)):
                ax.annotate("",
                            xy=(base_x, y_thick),
                            xytext=(base_x, y_thin),
                            arrowprops=dict(arrowstyle="->",
                                            color=color,
                                            lw=1.2,
                                            linestyle='--',
                                            alpha=0.8),
                            zorder=3)

    # Axis settings
    ax.set_xticks(x_pos)
    xtick_labels = []
    for i, c in conditions.iterrows():
        label = f"Condition {i+1}\nP={int(c['P(mbar)'])}, Ar={int(c['Ar(sccm)'])}"
        xtick_labels.append(label)
    ax.set_xticklabels(xtick_labels)
    
    ax.set_xlabel("Growth condition", labelpad=10)
    ax.set_ylabel("Phase fraction (%)", labelpad=10)
    ax.set_ylim(-5, 105)
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    
    # Grid: only horizontal dashed lines
    ax.grid(axis='y', alpha=0.35, linestyle='--', zorder=0)
    
    # Legend on the right
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0),
              frameon=False, ncol=1)
    
    plt.tight_layout()
    save_fig("Fig2_phase_fraction_by_condition")

# Execute plotting
if __name__ == "__main__":
    plot_phase_transition(df)
    print("Fig4 (phase fraction by condition) saved to DATA folder")
    print("Figure generated (PNG 1200dpi + PDF).")


Fig2 (phase fraction by condition) saved to DATA folder
Figure generated (PNG 1200dpi + PDF).


In [22]:
# -*- coding: utf-8 -*-
"""
精简版XRD三相竞争统计验证脚本
仅生成绘图所需核心CSV，数据源：DATA/XRD_final_results.csv
"""
import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel
from sklearn.linear_model import BayesianRidge

# -------------------------- 路径与数据读取 --------------------------
DATA_FOLDER = "DATA"
CSV_PATH = os.path.join(DATA_FOLDER, "XRD_final_results.csv")  # 固定数据源
os.makedirs(DATA_FOLDER, exist_ok=True)
RNG = np.random.default_rng(20261020)

# 列名映射 完全匹配当前CSV字段
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
RENAME = {
    "Ar(sccm)": "Ar",
    "P(mbar)": "P",
    "Si(μmol/min)": "Si",
    "Epitaxial growth time(s)": "Time",
    "Thickness groups": "Group",
    "Final RWP": "RWP",
    "α-phase fraction": "alpha",
    "β-phase fraction": "beta",
    "κ-phase fraction": "kappa",
}
df = raw.rename(columns=RENAME).copy()
df["g"] = (df["Time"] > 4000).astype(int)  # 0=薄(~100nm), 1=厚(~400nm)
FEAT = ["Ar", "P", "Si", "Time"]
X = df[FEAT].values

# 正则化Matern核
KERNEL = (C(1.0, (1e-2, 1e2))
          * Matern(1.5, length_scale_bounds=(1.0, 12.0), nu=2.5)
          + WhiteKernel(1e-3))

# -------------------------- 模型定义 --------------------------
def pls_gpr(nc):
    """StandardScaler -> PLS(nc) -> GPR 流水线"""
    def fit(Xtr, ytr, Xte):
        sc = StandardScaler().fit(Xtr)
        pl = PLSRegression(nc, scale=False)
        t = pl.fit_transform(sc.transform(Xtr), ytr)[0]
        gpr = GaussianProcessRegressor(kernel=KERNEL, alpha=1e-6, random_state=1).fit(t, ytr)
        return gpr.predict(pl.transform(sc.transform(Xte)))[0]
    return fit

def pls_bayes(nc):
    """StandardScaler -> PLS(nc) -> BayesianRidge 流水线"""
    def fit(Xtr, ytr, Xte):
        sc = StandardScaler().fit(Xtr)
        pl = PLSRegression(nc, scale=False)
        t = pl.fit_transform(sc.transform(Xtr), ytr)[0]
        b = BayesianRidge().fit(t, ytr)
        return b.predict(pl.transform(sc.transform(Xte)))[0]
    return fit

def bayes_ridge(Xtr, ytr, Xte):
    """直接在原始特征上做BayesianRidge"""
    sc = StandardScaler().fit(Xtr)
    b = BayesianRidge().fit(sc.transform(Xtr), ytr)
    return b.predict(sc.transform(Xte))[0]

# -------------------------- 统计指标函数 --------------------------
def loocv_q2(model, Xd, y):
    """留一法 Q² = 1 - PRESS/TSS"""
    n = len(y)
    pr = np.zeros(n)
    for i in range(n):
        m = np.arange(n) != i
        pr[i] = model(Xd[m], y[m], Xd[i:i + 1])
    return 1 - np.sum((y - pr) ** 2) / np.sum((y - y.mean()) ** 2)

def boot_q2(model, Xd, y, B=200):
    """Bootstrap 95%置信区间 for Q²"""
    n = len(y)
    v = []
    for _ in range(B):
        idx = RNG.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        try:
            v.append(loocv_q2(model, Xd[idx], y[idx]))
        except Exception:
            pass
    return np.percentile(v, [2.5, 50, 97.5])

def perm_two(a, b, B=20000):
    """两样本置换检验，返回均值差与双侧p值"""
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    obs = a.mean() - b.mean()
    pool = np.concatenate([a, b])
    n = len(a)
    ge = 0
    for _ in range(B):
        pm = RNG.permutation(pool)
        d = pm[:n].mean() - pm[n:].mean()
        if abs(d) >= abs(obs) - 1e-12:
            ge += 1
    return obs, (ge + 1) / (B + 1)

def cohen_d(a, b):
    """Cohen's d 标准化均值差"""
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1))
                 / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp

def bayes_posterior(y):
    """BayesianRidge 标准化系数 + 后验标准差"""
    sc = StandardScaler().fit(X)
    Xs = sc.transform(X)
    m = BayesianRidge().fit(Xs, y)
    prec = np.diag(np.full(Xs.shape[1], m.alpha_)) + m.lambda_ * (Xs.T @ Xs)
    sd = np.sqrt(np.diag(np.linalg.inv(prec)))
    return m.coef_, sd

# -------------------------- 主流程 --------------------------
def main():
    # 1. LOOCV Q² 四模型对比结果
    print("=== 计算 LOOCV Q² ===")
    models = [("PLS1-GPR", pls_gpr(1)), ("PLS2-GPR", pls_gpr(2)),
              ("PLS2-Bayes", pls_bayes(2)), ("BayesRidge", bayes_ridge)]
    q2_rows = []
    for r in ["kappa", "beta", "alpha"]:
        y = df[r].values
        qs = [loocv_q2(fn, X, y) for _, fn in models]
        ci = boot_q2(pls_bayes(2), X, y)
        print(f"{r:5s}: " + "  ".join(f"{nm} {q:+.2f}" for (nm, _), q in zip(models, qs)))
        row = {"response": r, "n": len(y)}
        for (nm, _), q in zip(models, qs):
            row[nm] = round(q, 3)
        row["CI2.5"] = round(ci[0], 3)
        row["CI97.5"] = round(ci[2], 3)
        q2_rows.append(row)
    pd.DataFrame(q2_rows).to_csv(
        os.path.join(DATA_FOLDER, "StatValidation_Q2.csv"),
        index=False, encoding="utf-8-sig")

    # 2. 贝叶斯后验系数
    print("\n=== 计算贝叶斯后验系数 ===")
    post_rows = []
    for r in ["kappa", "beta"]:
        coef, sd = bayes_posterior(df[r].values)
        print(f"{r:5s}: " + "  ".join(f"{f}={c:+.3f}±{s:.3f}" for f, c, s in zip(FEAT, coef, sd)))
        for f, c, s in zip(FEAT, coef, sd):
            post_rows.append({"response": r, "feature": f,
                              "coef": round(c, 4), "sd": round(s, 4)})
    pd.DataFrame(post_rows).to_csv(
        os.path.join(DATA_FOLDER, "StatValidation_posterior.csv"),
        index=False, encoding="utf-8-sig")

    # 3. 组间对比（压力对比 + 相分率薄厚对比）
    print("\n=== 组间置换检验 ===")
    contrast_rows = []
    def record(name, a, b, lab_a, lab_b):
        d, p = perm_two(a, b)
        dd = cohen_d(a, b)
        print(f"{name:28s}: {a.mean():.3f} vs {b.mean():.3f} diff={d:+.3f} p={p:.4f} d={dd:+.2f}")
        contrast_rows.append({"contrast": name, lab_a: round(a.mean(), 4),
                              lab_b: round(b.mean(), 4), "diff": round(d, 4),
                              "p": round(p, 4), "cohen_d": round(dd, 3)})

    # 压力对比 P20 vs P40
    for r in ["kappa", "beta", "alpha"]:
        record(f"P20 vs P40 ({r})", df[df.P == 20][r], df[df.P == 40][r], "P20", "P40")

    # 相分率 薄vs厚对比
    for r in ["kappa", "beta", "alpha"]:
        record(f"{r} thin vs thick", df[df.g == 0][r], df[df.g == 1][r], "thin", "thick")

    pd.DataFrame(contrast_rows).to_csv(
        os.path.join(DATA_FOLDER, "StatValidation_contrasts.csv"),
        index=False, encoding="utf-8-sig")
    print("\n统计验证完成，核心CSV已保存至 DATA/ 目录")

if __name__ == "__main__":
    main()


=== 计算 LOOCV Q² ===
kappa: PLS1-GPR +0.80  PLS2-GPR +0.91  PLS2-Bayes +0.89  BayesRidge +0.92
beta : PLS1-GPR +0.24  PLS2-GPR +0.03  PLS2-Bayes +0.21  BayesRidge +0.24
alpha: PLS1-GPR -0.83  PLS2-GPR -0.97  PLS2-Bayes -0.30  BayesRidge -0.39

=== 计算贝叶斯后验系数 ===
kappa: Ar=+0.021±0.045  P=-0.344±0.043  Si=-0.001±0.042  Time=+0.043±0.040
beta : Ar=-0.013±0.032  P=+0.170±0.029  Si=+0.036±0.028  Time=-0.054±0.026

=== 组间置换检验 ===
P20 vs P40 (kappa)          : 0.913 vs 0.114 diff=+0.799 p=0.0007 d=+9.12
P20 vs P40 (beta)           : 0.013 vs 0.502 diff=-0.489 p=0.0004 d=-3.05
P20 vs P40 (alpha)          : 0.073 vs 0.384 diff=-0.311 p=0.0081 d=-2.03
kappa thin vs thick         : 0.676 vs 0.751 diff=-0.076 p=0.6073 d=-0.20
beta thin vs thick          : 0.202 vs 0.069 diff=+0.134 p=0.3428 d=+0.50
alpha thin vs thick         : 0.122 vs 0.180 diff=-0.058 p=0.5992 d=-0.28

统计验证完成，核心CSV已保存至 DATA/ 目录


In [30]:
# -------------------------- 路径与全局样式 --------------------------
DATA_FOLDER = "DATA"
FIG_FOLDER = os.path.join(DATA_FOLDER, "StatValidation")
os.makedirs(FIG_FOLDER, exist_ok=True)

# 数据源文件
CSV_RAW = os.path.join(DATA_FOLDER, "XRD_final_results.csv")
CSV_Q2 = os.path.join(DATA_FOLDER, "StatValidation_Q2.csv")
CSV_POST = os.path.join(DATA_FOLDER, "StatValidation_posterior.csv")
CSV_CONTRAST = os.path.join(DATA_FOLDER, "StatValidation_contrasts.csv")

rng = np.random.default_rng(20261020)

# ========== 全局统一字号控制 ==========
LEGEND_FONT_SIZE = 12
AXIS_LABEL_SIZE = 14    # x/y轴标题（单位）字号
TICK_LABEL_SIZE = 12    # x/y轴刻度字号

# 期刊全局样式
plt.rcParams.update({
    "font.family": "Times New Roman",
    "axes.unicode_minus": False,
    "figure.dpi": 300,
    "savefig.dpi": 1200,
    "axes.formatter.useoffset": False,
    "font.size": 12,
    "axes.linewidth": 1.1,
    "axes.edgecolor": "#333333",

    # 刻度线加粗
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,

    # 坐标轴标题字号（全局统一）
    "axes.labelsize": AXIS_LABEL_SIZE,
    "axes.titlesize": 13,
    # 刻度字号（全局统一）
    "xtick.labelsize": TICK_LABEL_SIZE,
    "ytick.labelsize": TICK_LABEL_SIZE,

    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "font.weight": "normal",
})

# 三相统一配色
CA = "#c0392b"   # α 红
CB = "#27ae60"   # β 绿
CK = "#2980b9"   # κ 蓝
GREY = "#6b6b6b"

# -------------------------- 通用辅助函数：刻度文字加粗 --------------------------
def set_tick_bold(ax):
    """将指定坐标轴的x、y刻度标签全部加粗"""
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")
    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

# -------------------------- 读取所有数据 --------------------------
# 1. 原始XRD数据（用于GPR趋势图绘制）
raw = pd.read_csv(CSV_RAW, encoding="utf-8-sig")
RENAME = {
    "Ar(sccm)": "Ar", "P(mbar)": "P", "Si(μmol/min)": "Si",
    "Epitaxial growth time(s)": "Time", "Thickness groups": "Group",
    "Final RWP": "RWP", "α-phase fraction": "alpha",
    "β-phase fraction": "beta", "κ-phase fraction": "kappa",
}
df = raw.rename(columns=RENAME).copy()
df["g"] = (df["Time"] > 4000).astype(int)
FEAT = ["Ar", "P", "Si", "Time"]
X = df[FEAT].values

# GPR核函数 与统计脚本参数完全一致
KER = (C(1.0, (1e-2, 1e2))
       * Matern(1.5, length_scale_bounds=(1.0, 12.0), nu=2.5)
       + WhiteKernel(1e-3))

# 2. 统计结果CSV
df_q2 = pd.read_csv(CSV_Q2, encoding="utf-8-sig")
df_post = pd.read_csv(CSV_POST, encoding="utf-8-sig")
df_contrast = pd.read_csv(CSV_CONTRAST, encoding="utf-8-sig")

# 单图保存函数
def save_single(fig, name):
    fig.savefig(os.path.join(FIG_FOLDER, name + ".png"), dpi=1200,
                bbox_inches="tight", facecolor="white")
    fig.savefig(os.path.join(FIG_FOLDER, name + ".pdf"), bbox_inches="tight",
                facecolor="white")
    plt.close(fig)
    print(f"✅ 已保存: {name}")

# -------------------------- 仅保留GPR趋势计算辅助函数 --------------------------
def pls_gpr_transform(nc, Xtr, ytr, Xte):
    """PLS降维 + GPR拟合预测，与统计脚本逻辑完全一致"""
    sc = StandardScaler().fit(Xtr)
    pl = PLSRegression(nc, scale=False)
    t = pl.fit_transform(sc.transform(Xtr), ytr)[0]
    gpr = GaussianProcessRegressor(kernel=KER, alpha=1e-6, random_state=1).fit(t, ytr)
    t_te = pl.transform(sc.transform(Xte))
    return t.ravel(), gpr.predict(t_te, return_std=True), t_te.ravel()

# -------------------------- 1. κ相 PLS→GPR 趋势图 --------------------------
def fig_gpr_kappa():
    fig, ax = plt.subplots(figsize=(6.5, 5))
    y = df.kappa.values
    # 生成平滑趋势线
    t_grid = np.linspace(-2.2, 2.8, 300).reshape(-1, 1)
    sc = StandardScaler().fit(X)
    pl = PLSRegression(1, scale=False).fit(sc.transform(X), y)
    t_train = pl.transform(sc.transform(X)).ravel()
    gpr = GaussianProcessRegressor(kernel=KER, alpha=1e-6, random_state=1).fit(
        t_train.reshape(-1, 1), y)
    mu, std = gpr.predict(t_grid, return_std=True)

    ax.fill_between(t_grid.ravel(), mu - 1.96*std, mu + 1.96*std, color=CK, alpha=0.15)
    ax.plot(t_grid.ravel(), mu, color=CK, lw=2.2)

    # 散点：薄厚膜区分marker，颜色映射Si通量（×10⁻³单位）
    si_display = df.Si * 1000
    for gi, mk in [(0, "o"), (1, "s")]:
        idx = (df.g == gi).values
        scat = ax.scatter(t_train[idx], y[idx], c=si_display[idx], cmap="YlOrBr",
                          marker=mk, s=70, edgecolor="k", lw=0.7, zorder=5,
                          vmin=0, vmax=4)

    ax.set_xlabel(r"PLS latent variable $t_1$ (4 process variables)")
    ax.set_ylabel(r"$\kappa$-Ga$_2$O$_3$ phase fraction")
    ax.set_ylim(-0.08, 1.08)
    # 标题居中
    ax.set_title(r"$\kappa$ phase fraction: PLS$\to$GPR trend with 95% band")

    # 刻度加粗
    set_tick_bold(ax)

    # 图例：调用全局统一字号
    leg = [
        Line2D([], [], marker="o", color="w", markerfacecolor="gray", markeredgecolor="k", label="Thin film", ls=""),
        Line2D([], [], marker="s", color="w", markerfacecolor="gray", markeredgecolor="k", label="Thick film", ls=""),
    ]
    ax.legend(handles=leg, frameon=False, fontsize=LEGEND_FONT_SIZE,
          loc="center right", bbox_to_anchor=(1, 0.7),
          labelspacing=0.9, handletextpad=0.6, borderaxespad=0.8)

    # 色条 + 色条刻度加粗
    cbar = plt.colorbar(scat, ax=ax, label=r"SiH$_4$ flux ($\times10^{-3}\ \mu\mathrm{mol/min}$)")
    for label in cbar.ax.get_yticklabels():
        label.set_fontweight("bold")
    fig.tight_layout()
    save_single(fig, "Fig_GPR_kappa")

# -------------------------- 2. β相 PLS→GPR 趋势图 --------------------------
def fig_gpr_beta():
    fig, ax = plt.subplots(figsize=(6.5, 5))
    y = df.beta.values
    t_grid = np.linspace(-2, 2.5, 300).reshape(-1, 1)
    sc = StandardScaler().fit(X)
    pl = PLSRegression(1, scale=False).fit(sc.transform(X), y)
    t_train = pl.transform(sc.transform(X)).ravel()
    gpr = GaussianProcessRegressor(kernel=KER, alpha=1e-6, random_state=1).fit(
        t_train.reshape(-1, 1), y)
    mu, std = gpr.predict(t_grid, return_std=True)

    ax.fill_between(t_grid.ravel(), mu - 1.96*std, mu + 1.96*std, color=CB, alpha=0.15)
    ax.plot(t_grid.ravel(), mu, color=CB, lw=2.2)

    si_display = df.Si * 1000
    for gi, mk in [(0, "o"), (1, "s")]:
        idx = (df.g == gi).values
        scat = ax.scatter(t_train[idx], y[idx], c=si_display[idx], cmap="YlOrBr",
                          marker=mk, s=70, edgecolor="k", lw=0.7, zorder=5,
                          vmin=0, vmax=4)

    ax.set_xlabel(r"PLS latent variable $t_1$ (4 process variables)")
    ax.set_ylabel(r"$\beta$-Ga$_2$O$_3$ phase fraction")
    ax.set_ylim(-0.08, 1.08)
    # 标题居中
    ax.set_title(r"$\beta$ phase fraction: PLS$\to$GPR trend with 95% band")

    # 刻度加粗
    set_tick_bold(ax)

    # 图例：调用全局统一字号
    leg = [
        Line2D([], [], marker="o", color="w", markerfacecolor="gray", markeredgecolor="k", label="Thin film", ls=""),
        Line2D([], [], marker="s", color="w", markerfacecolor="gray", markeredgecolor="k", label="Thick film", ls=""),
    ]
    ax.legend(handles=leg, frameon=False, fontsize=LEGEND_FONT_SIZE,
          loc="center right", bbox_to_anchor=(1, 0.7),
          labelspacing=0.9, handletextpad=0.6, borderaxespad=0.8)

    # 色条 + 色条刻度加粗
    cbar = plt.colorbar(scat, ax=ax, label=r"SiH$_4$ flux ($\times10^{-3}\ \mu\mathrm{mol/min}$)")
    for label in cbar.ax.get_yticklabels():
        label.set_fontweight("bold")
    fig.tight_layout()
    save_single(fig, "Fig_GPR_beta")

# -------------------------- 3. LOOCV Q² 四模型对比 --------------------------
def fig_q2_comparison():
    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    # CSV实际列名，与显示标签分开映射
    model_cols = ["PLS1-GPR", "PLS2-GPR", "PLS2-Bayes", "BayesRidge"]
    model_labels = ["PLS1-GPR", "PLS2-GPR", "PLS2-Bayes", "Bayes-Ridge"]
    xp = np.arange(len(model_cols))
    w = 0.24  # 微调柱宽，组内更紧凑

    phase_info = [("kappa", CK), ("beta", CB), ("alpha", CA)]
    for k, (r, col) in enumerate(phase_info):
        row = df_q2[df_q2["response"] == r].iloc[0]
        qs = [row[m] for m in model_cols]
        ax.bar(xp + (k - 1) * w, qs, w, color=col, edgecolor="white", label=f"{r}-phase")

    ax.axhline(0, color="#333", lw=0.8)
    ax.set_xticks(xp)
    # x轴标签水平放置
    ax.set_xticklabels(model_labels, rotation=0, ha="center")
    ax.set_ylabel(r"LOOCV Q$^2$")
    # 标题居中
    ax.set_title("Predictive performance (LOOCV Q²) by phase and model")

    # 刻度加粗
    set_tick_bold(ax)

    # 图例：调用全局统一字号
    ax.legend(
        frameon=False,
        loc="lower right",
        ncol=1,
        fontsize=LEGEND_FONT_SIZE,
        labelspacing=0.8,
        handletextpad=0.5,
        borderaxespad=0.5
    )

    ax.set_ylim(-1, 1)
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    fig.tight_layout()
    save_single(fig, "Fig_Q2_four_models")

# -------------------------- 4. 贝叶斯后验系数 --------------------------
def fig_bayesian_posterior():
    fig, ax = plt.subplots(figsize=(7, 5.2))
    feat_order = ["Ar", "P", "Si", "Time"]
    xx = np.arange(len(feat_order))
    ww = 0.36

    # κ相
    k_data = df_post[df_post["response"] == "kappa"].set_index("feature").reindex(feat_order)
    ax.barh(xx - ww/2, k_data["coef"], ww, xerr=1.96*k_data["sd"],
            color=CK, ecolor=CK, alpha=0.85, capsize=2, label=r"$\kappa$ fraction")

    # β相
    b_data = df_post[df_post["response"] == "beta"].set_index("feature").reindex(feat_order)
    ax.barh(xx + ww/2, b_data["coef"], ww, xerr=1.96*b_data["sd"],
            color=CB, ecolor=CB, alpha=0.85, capsize=2, label=r"$\beta$ fraction")

    ax.axvline(0, color="#333", lw=0.8)
    ax.set_yticks(xx)
    ax.set_yticklabels(feat_order)
    ax.set_xlabel("Standardized posterior coefficient (95% CrI)")
    # 标题居中
    ax.set_title("Bayesian variable attribution")

    # 刻度加粗
    set_tick_bold(ax)

    # 图例：调用全局统一字号
    ax.legend(frameon=False, fontsize=LEGEND_FONT_SIZE, loc="upper right",
              labelspacing=0.8, handletextpad=0.5)
    fig.tight_layout()
    save_single(fig, "Fig_Bayesian_coefficients")

# -------------------------- 5. κ相压力坍缩 --------------------------
def fig_kappa_pressure_collapse():
    fig, ax = plt.subplots(figsize=(5.8, 5.2))
    sub_p20 = df[(df.Ar == 80) & (df.P == 20)]
    sub_p40 = df[(df.Ar == 80) & (df.P == 40)]

    # 带抖动散点
    ax.scatter(np.full(len(sub_p20), 0) + rng.uniform(-0.04, 0.04, len(sub_p20)),
               sub_p20.kappa, color=CK, edgecolor="k", s=65, zorder=5)
    ax.scatter(np.full(len(sub_p40), 1) + rng.uniform(-0.04, 0.04, len(sub_p40)),
               sub_p40.kappa, color=CB, edgecolor="k", s=65, zorder=5)

    # 均值标记
    ax.scatter(0, sub_p20.kappa.mean(), marker="_", s=450, color="k", lw=2)
    ax.scatter(1, sub_p40.kappa.mean(), marker="_", s=450, color="k", lw=2)

    # 从CSV读取统计结果并标注
    row = df_contrast[df_contrast["contrast"] == "P20 vs P40 (kappa)"].iloc[0]
    p_val = row["p"]
    d_val = row["cohen_d"]

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["P=20 mbar\n(Ar=80 fixed)", "P=40 mbar\n(Ar=80 fixed)"])
    ax.set_ylabel(r"$\kappa$-Ga$_2$O$_3$ phase fraction")
    ax.set_ylim(-0.05, 1.08)
    # 标题居中
    ax.set_title(r"$\kappa$ phase collapse: P20 $\to$ P40 (fixed Ar=80)")

    # 刻度加粗
    set_tick_bold(ax)

    # 趋势箭头 + 统计量标注
    ax.annotate("", xy=(1, sub_p40.kappa.mean()+0.06), xytext=(0, sub_p20.kappa.mean()-0.06),
                arrowprops=dict(arrowstyle="->", color=GREY, lw=1.4))
    ax.text(0.5, 0.08, f"$p={p_val:.4f}$\nCohen's $d={d_val:.1f}$",
            transform=ax.transAxes, ha="center", fontsize=10,
            bbox=dict(fc="white", ec="#cccccc", boxstyle="round,pad=0.3"))
    fig.tight_layout()
    save_single(fig, "Fig_kappa_pressure_collapse")

# -------------------------- 6. 薄厚膜三相对比 --------------------------
def fig_phase_thickness_contrast():
    fig, ax = plt.subplots(figsize=(7, 5.2))
    phases = [("alpha", CA), ("beta", CB), ("kappa", CK)]
    xp = np.arange(len(phases))
    w = 0.3

    for i, (ph, color) in enumerate(phases):
        row = df_contrast[df_contrast["contrast"] == f"{ph} thin vs thick"].iloc[0]
        v_thin = row["thin"]
        v_thick = row["thick"]
        p_val = row["p"]

        # 柱状图
        ax.bar(xp[i] - w/2, v_thin*100, w, color="white", edgecolor=color, lw=1.5, label=f"{ph} thin" if i == 0 else "")
        ax.bar(xp[i] + w/2, v_thick*100, w, color=color, edgecolor="black", lw=0.8, label=f"{ph} thick" if i == 0 else "")

        # 显著性标记
        if p_val < 0.05:
            sig = "*"
        elif p_val < 0.1:
            sig = "†"
        else:
            sig = "ns"
        y_max = max(v_thin, v_thick)*100 + 3
        ax.text(xp[i], y_max, sig, ha="center", fontsize=12, fontweight="bold")

    ax.set_xticks(xp)
    ax.set_xticklabels([r"$\alpha$-phase", r"$\beta$-phase", r"$\kappa$-phase"])
    ax.set_ylabel("Phase fraction (%)")
    ax.set_ylim(0, 105)
    # 标题居中
    ax.set_title("Phase fraction: thin vs thick film")

    # 刻度加粗
    set_tick_bold(ax)

    # 图例：调用全局统一字号
    ax.legend(frameon=False, ncol=2, loc="upper right",
              fontsize=LEGEND_FONT_SIZE, labelspacing=0.8, columnspacing=1.0)

    ax.grid(axis='y', alpha=0.25, linestyle='--')
    fig.tight_layout()
    save_single(fig, "Fig_phase_thickness_contrast")

# -------------------------- 执行全部绘图 --------------------------
if __name__ == "__main__":
    fig_gpr_kappa()
    fig_gpr_beta()
    fig_q2_comparison()
    fig_bayesian_posterior()
    fig_kappa_pressure_collapse()
    fig_phase_thickness_contrast()
    print("\n🎉 所有统计图表生成完成，已保存至 DATA/StatValidation/ 目录")


✅ 已保存: Fig_GPR_kappa
✅ 已保存: Fig_GPR_beta
✅ 已保存: Fig_Q2_four_models
✅ 已保存: Fig_Bayesian_coefficients
✅ 已保存: Fig_kappa_pressure_collapse
✅ 已保存: Fig_phase_thickness_contrast

🎉 所有统计图表生成完成，已保存至 DATA/StatValidation/ 目录


In [31]:
# =============================================================================
# Plot phase fraction scatter plot with arrows for phase transition visualization
# Data source: XRD_final_results.csv stored in ./DATA
# Output: PNG and vector PDF figures saved to ./DATA
# Consistent font size & bold style aligned with Figure 3 for journal submission
# =============================================================================
warnings.filterwarnings("ignore")

# -------------------------- Journal Figure Style Settings --------------------------
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 1200
plt.rcParams['axes.linewidth'] = 0.8

# Font size & bold setting, fully consistent with Fig.3
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 17
plt.rcParams['ytick.labelsize'] = 17
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# -------------------------- Relative File Path Configuration --------------------------
DATA_FOLDER = "DATA"
csv_path = os.path.join(DATA_FOLDER, "XRD_final_results.csv")
save_png = os.path.join(DATA_FOLDER, "phase_fraction_correct_ratio.png")
save_pdf = os.path.join(DATA_FOLDER, "phase_fraction_correct_ratio.pdf")

# Load experimental dataset
df = pd.read_csv(csv_path, encoding="utf-8-sig")

# Definition of four growth conditions
cond_def = [
    {"Ar":40, "P":20, "base_x":0, "label":"Condition 1\nP=20, Ar=40"},
    {"Ar":60, "P":20, "base_x":1, "label":"Condition 2\nP=20, Ar=60"},
    {"Ar":80, "P":20, "base_x":2, "label":"Condition 3\nP=20, Ar=80"},
    {"Ar":80, "P":40, "base_x":3, "label":"Condition 4\nP=40, Ar=80"},
]

# Six sample categories within each growth condition
cat_list = [
    {"key":"α_thin",  "phase":"α", "thick":"Thin",  "marker":"o", "color":"#d62728"},
    {"key":"α_thick", "phase":"α", "thick":"Thick", "marker":"s", "color":"#d62728"},
    {"key":"κ_thin",  "phase":"κ", "thick":"Thin",  "marker":"o", "color":"#1f77b4"},
    {"key":"κ_thick", "phase":"κ", "thick":"Thick", "marker":"s", "color":"#1f77b4"},
    {"key":"β_thin",  "phase":"β", "thick":"Thin",  "marker":"o", "color":"#2ca02c"},
    {"key":"β_thick", "phase":"β", "thick":"Thick", "marker":"s", "color":"#2ca02c"},
]

col_spacing = 0.13
phase_col_map = {
    "α": "α-phase fraction",
    "κ": "κ-phase fraction",
    "β": "β-phase fraction",
}

# Canvas dimension: aspect ratio ~2.31, compatible with journal single-column / wide figure
fig, ax = plt.subplots(figsize=(20.8, 9))
y_min, y_max = -6, 106

for cond in cond_def:
    ar_val = cond["Ar"]
    p_val = cond["P"]
    base_x = cond["base_x"]
    # Filter data by Ar flow and chamber pressure
    df_cond = df[(df["Ar(sccm)"] == ar_val) & (df["P(mbar)"] == p_val)].copy()

    for cat_idx, cat in enumerate(cat_list):
        cat_x = base_x + (cat_idx - 2.5) * col_spacing
        ph = cat["phase"]
        thick_type = cat["thick"]
        mk = cat["marker"]
        c = cat["color"]
        # Filter thin/thick samples
        df_sub = df_cond[df_cond["Thickness groups"].str.contains(thick_type, na=False)]
        if len(df_sub) == 0:
            continue

        all_points = []
        # Undoped reference point (Si flux = 0)
        row_0 = df_sub[df_sub["Si(μmol/min)"] == 0]
        if len(row_0) == 0:
            continue
        y0 = row_0[phase_col_map[ph]].iloc[0] * 100
        x0 = cat_x - 0.025
        all_points.append({
            "y": y0, "x": x0,
            "is_doped": False, "row": row_0.iloc[0]
        })

        # Doped samples with Si incorporation
        doped_rows = df_sub[df_sub["Si(μmol/min)"] > 0].reset_index(drop=True)
        for d_idx, (_, row_d) in enumerate(doped_rows.iterrows()):
            yd = row_d[phase_col_map[ph]] * 100
            xd = cat_x + 0.025
            all_points.append({
                "y": yd, "x": xd,
                "is_doped": True, "row": row_d
            })

        # Sort points by phase fraction value
        all_points.sort(key=lambda p: p["y"])

        # Plot undoped reference marker (hollow symbol)
        ax.scatter(x0, y0, marker=mk, facecolors="white",
                   edgecolors=c, s=90, linewidth=2, zorder=4)

        # Plot doped markers and transition arrows
        for p_idx, p in enumerate(all_points):
            if not p["is_doped"]:
                continue
            yd = p["y"]
            xd = p["x"]
            ax.scatter(xd, yd, marker=mk, facecolors=c,
                       edgecolors=c, s=90, linewidth=2, zorder=4)
            # Dashed arrow to visualize phase evolution upon Si doping
            arrow = FancyArrowPatch((x0, y0), (xd, yd),
                                    arrowstyle="->", color=c,
                                    linestyle="--", linewidth=1.2,
                                    mutation_scale=15, alpha=0.9, zorder=2)
            ax.add_patch(arrow)

# Axes configuration
ax.set_xticks([c["base_x"] for c in cond_def])
ax.set_xticklabels([c["label"] for c in cond_def])
ax.set_ylabel("Phase fraction (%)")
ax.set_xlabel("Growth condition")
ax.set_ylim(y_min, y_max)
ax.grid(axis="y", linestyle="--", alpha=0.45)

# Custom legend definition
legend_elements = [
    Line2D([],[],marker="o",mfc="white",mec="#d62728",linestyle="",markersize=9,label=r'$\alpha$-phase (thin)'),
    Line2D([],[],marker="s",mfc="white",mec="#d62728",linestyle="",markersize=9,label=r'$\alpha$-phase (thick)'),
    Line2D([],[],marker="o",mfc="white",mec="#1f77b4",linestyle="",markersize=9,label=r'$\kappa$-phase (thin)'),
    Line2D([],[],marker="s",mfc="white",mec="#1f77b4",linestyle="",markersize=9,label=r'$\kappa$-phase (thick)'),
    Line2D([],[],marker="o",mfc="white",mec="#2ca02c",linestyle="",markersize=9,label=r'$\beta$-phase (thin)'),
    Line2D([],[],marker="s",mfc="white",mec="#2ca02c",linestyle="",markersize=9,label=r'$\beta$-phase (thick)'),
    Line2D([],[],marker="o",mfc="white",mec="black",linestyle="",markersize=9,label="Undoped"),
    Line2D([],[],marker="o",mfc="black",mec="black",linestyle="",markersize=9,label="Doped"),
]
ax.legend(handles=legend_elements, loc="upper left", bbox_to_anchor=(1.01, 1),
          frameon=False, labelspacing=0.8)

plt.tight_layout()
# Save figure
plt.savefig(save_png, bbox_inches="tight", facecolor="white")
plt.savefig(save_pdf, bbox_inches="tight", facecolor="white")
plt.close()

print(f"✅ Plot finished. Annotations of ε and Si values removed.")
print(f"Font size & bold style matched with Fig.3.")
print(f"Figure size: width 20.8 × height 9, aspect ratio ≈2.31")
print(f"Output files:")
print(f"  {save_png}")
print(f"  {save_pdf}")


✅ Plot finished. Annotations of ε and Si values removed.
Font size & bold style matched with Fig.3.
Figure size: width 20.8 × height 9, aspect ratio ≈2.31
Output files:
  DATA\phase_fraction_correct_ratio.png
  DATA\phase_fraction_correct_ratio.pdf


In [ ]:
# ---- Text for Supporting Information / GitHub README ----
"""
All XRD raw data, processing code, and fitted figures are available at the GitHub repository:
https://github.com/xxx/xxx

The analysis workflow is implemented in Jupyter notebooks.
The script performs multi-peak Voigt fitting for overlapping XRD peaks of α/β/κ-Ga₂O₃,
calculates phase fractions, crystallite size (Scherrer equation), and lattice microstrain.
All figures are exported as vector PDF and 1200 dpi PNG for journal submission.
The summary table of fitted parameters is exported as CSV.
"""
